In [1]:
from jugaad_data.nse import NSELive
n = NSELive()
status = n.market_status()
status['marketState']

[{'market': 'Capital Market',
  'marketStatus': 'Closed',
  'tradeDate': '12-Nov-2025 15:30',
  'index': 'NIFTY 50',
  'last': 25875.8,
  'variation': 180.84999999999854,
  'percentChange': 0.7,
  'marketStatusMessage': 'Normal Market has Closed'},
 {'market': 'Currency',
  'marketStatus': 'Closed',
  'tradeDate': '12-Nov-2025',
  'index': '',
  'last': '',
  'variation': '',
  'percentChange': '',
  'marketStatusMessage': 'Market is Closed'},
 {'market': 'Commodity',
  'marketStatus': 'Open',
  'tradeDate': '12-Nov-2025',
  'index': '',
  'last': '',
  'variation': '',
  'percentChange': '',
  'marketStatusMessage': 'Market is Open'},
 {'market': 'Debt',
  'marketStatus': 'Closed',
  'tradeDate': '12-Nov-2025',
  'index': '',
  'last': '',
  'variation': '',
  'percentChange': '',
  'marketStatusMessage': 'Market is Closed'},
 {'market': 'currencyfuture',
  'marketStatus': 'Closed',
  'tradeDate': '12-Nov-2025',
  'index': '',
  'last': '88.6875',
  'variation': '',
  'percentChange':

In [2]:
tick_data = n.tick_data("HDFC")

In [3]:
tick_data['grapthData'][0:10]


[]

In [4]:
option_chain = n.index_option_chain("NIFTY") # Index Option chains
eq_option_chain = n.equities_option_chain("RELIANCE") # Equity option chains
curr_option_chain = n.currency_option_chain("USDINR") # Currency option chains

In [5]:
option_chain['records']['underlyingValue']

25875.8

In [6]:
testDF = option_chain['filtered']['data']

In [ ]:
testDF

In [ ]:
count =0 
for i in testDF:
    if(i['CE']['openInterest']>50000 or i['PE']['openInterest']>50000): 
        print(i['strikePrice'])
        count=count+1
print(count)

In [ ]:

for option in testDF :
    print( "{}\t{}\t::\t{}\t::\t{}\t{}".format(option['CE']['openInterest'],option['CE']['lastPrice'], option['strikePrice'], option['PE']['lastPrice'],option['PE']['openInterest']))
    # print(option['CE']['openInterest'])



In [ ]:
from helpers.nse_data import NSEData
from helpers.FnO import analyse_option_chain, get_next_expiry_date

In [ ]:
def myplotsOptionChain(r):
    df_data = {'contract_type':[],'expiryDate': [],'strikePrice':[],'openInterest': [],'changeinOpenInterest': [],'pchangeinOpenInterest': [],'totalTradedVolume': [], 
               'totalBuyQuantity': [], 'totalSellQuantity': [], 'change':[]}
    keys = df_data.keys()
    data = r['records']['data']

    for entry in data:
        if entry.get('CE'):
            [df_data[name].append(entry['CE'][name]) if name!='contract_type' else df_data['contract_type'].append('Calls_CE') for name in keys]

        if entry.get('PE'):
            [df_data[name].append(entry['PE'][name]) if name!='contract_type' else df_data['contract_type'].append('Puts_PE') for name in keys]
        

    df = pd.DataFrame(df_data)
    df.rename(columns = {'expiryDate':'expiry_date', 'strikePrice':'strike_price'},inplace = True)
    df['expiry_date'] = df['expiry_date'].apply(lambda x:datetime.strptime(x, "%d-%b-%Y").strftime("%d-%b-%Y"))
    df['strike_price'] = df['strike_price'].apply(lambda x: int(x))
    df['absChangeOI'] = df['changeinOpenInterest'].apply(lambda x: abs(x))
    df['absChange'] = df['change'].apply(lambda x: abs(x))
    
    expiry_dates= ['30-Apr-2025']
    # Get specific expiry date
    if not expiry_dates:
        recent_expiry = get_next_expiry_date()[0]
        sup_plot_text_date = recent_expiry
        expiry_dates = [recent_expiry] # Single element tuple requires ,
    else:
        recent_expiry = expiry_dates[0]
        sup_plot_text_date = 'All Available Expiries'
        expiry_dates = [datetime.strptime(x, "%d-%b-%Y").strftime("%d-%b-%Y") for x in expiry_dates]   
    df = df[df['expiry_date'].isin(expiry_dates)]
    
    Plots.plot_Option_chain("symbol", df, compare_with, top_n, sup_plot_text_date, fig_size=fig_size)

    return df

In [ ]:
import pandas as pd
from datetime import datetime
from helpers.plotting import Plots 
import matplotlib.pyplot as plt
import seaborn as sns
myplotsOptionChain(option_chain)

In [ ]:
  analyse_option_chain('TATACHEM', plot = True, fig_size=(21,6.7)) # Read the Doc String. Manually compare other parameters such as openInterest